### ORR Station entry and exit numbers  by station and LA

2003/04 data had to be interpolated. This was done by taking the midpoint between 2002/03 and 2004/05. This was done directly in the Excel file.

I've also had to add the ONS LA codes for ease when joining the data. This was also done directly in the Excel file.

In [50]:
import pandas as pd

In [51]:
raw_orr_data = pd.read_excel("../../data/station entries and exits.xlsx", sheet_name="1415a_Entries_and_Exits", skiprows = 3, na_values = ['[z]', '[x]'])

In [52]:
raw_orr_data = raw_orr_data[~raw_orr_data['Region'].isin(["Scotland", "[z]"])& ~raw_orr_data['Region'].isna()]



In [53]:
raw_orr_data = pd.melt(
    raw_orr_data,
    id_vars=[col for col in raw_orr_data.columns if not pd.Series(col).str.match(r'^\d{4}/\d{2}$')[0]],   # columns to keep fixed
    value_vars=raw_orr_data.filter(regex=r'^\d{4}/\d{2}$').columns,    # columns to unpivot (optional)
    var_name='Financial year',                    # name for new variable column
    value_name='Value'                      # name for new value column
)

In [54]:
orr_data = raw_orr_data.groupby(['Financial year', 'Local authority: district or unitary', 'Local authority code'], as_index=False)['Value'].sum()

In [55]:
fy_lookup = pd.read_csv("../../data/Financial year lookup.csv")

fy_lookup['year_month'] = pd.to_datetime(fy_lookup['year_month'], format='%d/%m/%Y')

In [56]:
orr_data

,Financial year,Local authority: district or unitary,Local authority code,Value
0,1997/98,Adur,E07000223,1797095.0
1,1997/98,Amber Valley,E07000032,315245.0
2,1997/98,Arun,E07000224,2306570.0
3,1997/98,Ashfield,E07000170,364333.0
4,1997/98,Ashford,E07000105,2072317.0
...,...,...,...,...
8365,2023/24,Wrexham,W06000006,870868.0
8366,2023/24,Wychavon,E07000238,1305294.0
8367,2023/24,Wyre,E07000128,582988.0
8368,2023/24,Wyre Forest,E07000239,1216566.0


In [57]:
orr_data = pd.merge(orr_data, fy_lookup, how='outer', on='Financial year')

### GDP data

In [58]:
gdp_data = pd.read_excel("../../data/monthlygdpto4dp.xlsx", sheet_name="Data_table", skiprows=3)

In [59]:
gdp_data['Month'] = pd.to_datetime(gdp_data['Month'], format='%Y%b')

In [60]:
gdp_data = gdp_data[['Month', 'Monthly GDP (A-T)']]
gdp_data.rename(columns={'Monthly GDP (A-T)': 'GDP'}, inplace=True)

In [61]:
gdp_orr_data = pd.merge(orr_data, gdp_data, how='left', left_on='year_month', right_on='Month')

### CPIH

In [62]:
cpih = pd.read_csv("../../data/cpih.csv", skiprows=189)
cpih.rename(columns={'2025 Q2': 'year_month', '4.1': 'CPIH'}, inplace=True)
cpih['year_month'] = pd.to_datetime(cpih['year_month'], format='%Y %b')


In [63]:
cpih_gdp_orr_data = pd.merge(gdp_orr_data, cpih, how='left', on='year_month')

In [ ]:
cpih_gdp_orr_data

,Financial year,Local authority: district or unitary,Local authority code,Value,year_month,Month,GDP,CPIH
0,1997/98,Adur,E07000223,1797095.0,1997-04-01,1997-04-01,63.2687,2.1
1,1997/98,Adur,E07000223,1797095.0,1997-05-01,1997-05-01,62.6762,2.1
2,1997/98,Adur,E07000223,1797095.0,1997-06-01,1997-06-01,62.9773,2.1
3,1997/98,Adur,E07000223,1797095.0,1997-07-01,1997-07-01,63.3714,2.4
4,1997/98,Adur,E07000223,1797095.0,1997-08-01,1997-08-01,63.3912,2.3
...,...,...,...,...,...,...,...,...
100435,2023/24,York,E06000014,9274308.0,2023-11-01,2023-11-01,100.2761,4.2
100436,2023/24,York,E06000014,9274308.0,2023-12-01,2023-12-01,100.2354,4.2
100437,2023/24,York,E06000014,9274308.0,2024-01-01,2024-01-01,100.7388,4.2
100438,2023/24,York,E06000014,9274308.0,2024-02-01,2024-02-01,100.9559,3.8
